In [1]:
# mike babb
# created: 2026 08 23
# updated: 2026 09 23
# find five words with 25 different letters
# example: ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## DEMONSTRATE BITWISE OPERATIONS

In [7]:
vibex = byte_encode_words('vibex')
glyph = byte_encode_words('glyph')
muntz = byte_encode_words('muntz')
dwarf = byte_encode_words('dwarf')
jocks = byte_encode_words('jocks')
cramp = byte_encode_words('cramp')

In [11]:
# this is equal to zero - no letters reused
(vibex | glyph | muntz | dwarf) & jocks 

0

In [12]:
# this is not equal to zero because letters are reused
((vibex | glyph) | muntz | dwarf) & cramp

167937

In [13]:
# order of operations for bitwise operations
(vibex | glyph) & (muntz | dwarf) 

0

In [14]:
vibex | glyph | muntz | dwarf | jocks

67043327

In [15]:
# same output as above
byte_encode_words('vibexglyphmuntzdwarfjocks')

67043327

# BUILD LEVEL 2 BY COMBINING TWO BYTE ENCODED WORDS

In [16]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common, compute the bitwise or to add the words together
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])
l2_df.shape

(3213696, 3)

# BUILD LEVELS 4 AND 5 BY COMBINING TWO ITEMS FROM THE L2 LIST  
# COMPARE THAT WITH THE WORD BYTE ARRAY ONE MORE TIME

In [19]:
# get words from bytes
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [18]:
w_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [20]:
l2_all = w_l2_df['l2'].to_numpy(dtype = np.int32)

In [21]:
# let's just use the word jocks
w_l2_df = w_l2_df.loc[(w_l2_df['w1'] == 'jocks') |
                              (w_l2_df['w2'] == 'jocks'), ['w1b', 'w2b', 'l2']].reset_index(drop = True)

In [22]:
w_l2_df.shape

(928, 3)

In [23]:
w_l2_df['l2'].unique().shape

(928,)

In [24]:
l2_all.shape

(640023,)

In [25]:
# we are going to make a lot of comparisons
print('The full set of l2 - duplicated l2:', l2_df.shape[0], l2_df.shape[0] ** 2)
print('The unique l2:', l2_df['l2'].unique().shape[0],  l2_df['l2'].unique().shape[0]** 2)
# but, we'll be clever about this and compare each item from the l2_list
# against the whole l2_list using array operations. 


The full set of l2 - duplicated l2: 3213696 10327841980416
The unique l2: 640023 409629440529


# COMPUTE THE COMBINATIONS

In [51]:
w_l2_df.shape

(928, 3)

In [52]:
l2_all.shape

(640023,)

In [54]:
w_l2_df.head()

,w1b,w2b,l2
0,8219,280068,288287
1,283,280068,280351
2,4371,280068,284439
3,2075,280068,282143
4,133139,280068,413207


In [71]:
testo = ((l2_all | l2_all[10]) & 1484256) == 0
testo.sum()


np.int64(195)

In [55]:
positional_idx_l2l3l4 = (l2_all & l2_all[10]) == 0 

In [ ]:

positional_idx_l2l3l4 = (l2_all & l2_all[10]) == 0 
output_array_w3bw4b = l2_all[positional_idx_l2l3l4]
#print(output_array_w3bw4b)
output_array_l2l3l4 = output_array_w3bw4b | l2
#print(output_array_l2l3l4)
n_rows_l2l3l4 = output_array_l2l3l4.shape[0]
#print(n_rows_l2l3l4)

for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):
    # w3bw4b: this is the other l2. l2 and w3bw4b have no letters in common. 20 letters.

    # create an indexer
    print(w3bw4b)
    positional_idx_l2l3l4l5 = (word_byte_array & w3bw4b) == 0
    print(positional_idx_l2l3l4l5.sum())
    temp_output = np.zeros(shape = ())




[1484256 1484736 3581376 7774656]
[51836900 51836868 53934020 58128324]
4
1484256
236
1484736
264
3581376
242
7774656
249


In [110]:
# compute all possible pairs
start_pos = 0
total_output = np.zeros(shape = (100_000_000, 5), dtype = np.int32)
for i_row, row in w_l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row
    # print(w1b, w2b, l2)

    # compare the current l2 to all l2 - this will find all instances
    # a value of zero indicates that there are no letters in common
    # indexer for l2, l3, and l4
    # bitwise and to identify two l2 that do not have a letter in common
    positional_idx_l2l3l4 = (l2_all & l2) == 0 
    #print(positional_idx_l2l3l4.shape)
    # TODO: fix this!
    if positional_idx_l2l3l4.any():

        # these are l2 words with different letters the the l2 above. 
        # this is effectively l4
        # this is now 20 different letters
        output_array_w3bw4b = l2_all[positional_idx_l2l3l4]
        # print(output_array_w3bw4b.shape)
        
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = output_array_w3bw4b | l2

        # create the temp output
        n_rows_l2l3l4 = output_array_l2l3l4.shape[0]
        # print(n_rows_l2l3l4)

        # check against the word_byte_array for w5b        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):
            # w3bw4b: this is the other l2. l2 and w3bw4b have no letters in common. 20 letters.

            # create an indexer over bitwise and
            positional_idx_l2l3l4l5 = (word_byte_array & w3bw4b) == 0

            if positional_idx_l2l3l4l5.any(): 
                # the indexer has at least one True value.
                
                # output_array_l2l3l4l5 is the list of word(s) that have letters
                # that do not match the other letters. In other words, this is 
                # the final five letters not in the group of twenty
                output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]                
                # print(output_array_l2l3l4l5.shape)

                # count!
                n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]

                # create a temporary matrix to hold the output
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                # word 1
                temp_output[:, 0] = w1b
                # word 2
                temp_output[:, 1] = w2b
                # the bitwise or on w1 and w2
                temp_output[:, 2] = l2

                # calculate w3b and w4b
                # this is the 'other' w1b and w2b values - w3b and w4b, effectively.
                temp_output[:, 3] = w3bw4b
                # the final word
                temp_output[:, 4] = output_array_l2l3l4l5

                # perform another round of operations
                l2l3l4l5_bitwise = ((temp_output[:, 2] | temp_output[:, 3]) & temp_output[:, 4]) == 0
                if l2l3l4l5_bitwise.any():
                    temp_output = temp_output[l2l3l4l5_bitwise, :]
                    n_rows_l2l3l4l5 = temp_output.shape[0]

                    # update the total output with the temporary list
                    total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output

                    # the counter
                    start_pos += n_rows_l2l3l4l5
                    # print(start_pos) 
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row)   

0


In [ ]:
output_df = pd.DataFrame(data = total_output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])

In [85]:
output_df.shape

(180, 5)

In [86]:
output_df['l2l3l4_bitwise_and'] = output_df['l2'] & output_df['l3l4']
output_df['l2l3l4_bitwise_or'] = output_df['l2'] | output_df['l3l4']

In [87]:
output_df['l2l3l4l5_bitwise_and'] = output_df['l2l3l4_bitwise_or'] & output_df['w5b']

In [88]:
output_df['l2l3l4l5_bitwise_and'].value_counts()

l2l3l4l5_bitwise_and
282880      4
1284        4
24832       3
271360      3
24576       3
           ..
25344       1
21248       1
270596      1
288000      1
50344192    1
Name: count, Length: 136, dtype: int64

# CREATE AND SHAPE THE OUTPUT

In [89]:
output = total_output[:start_pos]

In [90]:
output.shape

(67, 5)

In [91]:
# turn it into a dataframe
output_df = pd.DataFrame(data = output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])

In [92]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b
0,33562898,280068,33842966,27920489,5279872
1,33562898,280068,33842966,14194856,19005505
2,33562898,280068,33842966,24285377,8914984
3,33685523,280068,33965591,27797864,5279872
4,33685523,280068,33965591,14194856,18882880


In [93]:
output_df['l2'].unique().shape

(15,)

In [94]:
output_df['l3l4'].unique().shape

(35,)

In [95]:
check = output_df.drop_duplicates(subset = ['w1b', 'w2b'])
check.shape

(15, 5)

In [96]:
check.shape

(15, 5)

# JOIN TO GET THE W3B AND THE W4B

In [97]:
l3l4_df = l2_df[['w1b', 'w2b', 'l2']].copy()
l3l4_df.columns = ['w3b', 'w4b', 'l3l4',]

In [98]:
l3l4_df.shape

(3213696, 3)

In [99]:
output_df = pd.merge(left = output_df, right = l3l4_df)

In [100]:
output_df.shape

(81, 7)

In [101]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b,w3b,w4b
0,33562898,280068,33842966,27920489,5279872,8914984,19005505
1,33562898,280068,33842966,14194856,19005505,8914984,5279872
2,33562898,280068,33842966,24285377,8914984,19005505,5279872
3,33685523,280068,33965591,27797864,5279872,8914984,18882880
4,33685523,280068,33965591,14194856,18882880,8914984,5279872


In [102]:
# reorder...
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b']
output_df = output_df[col_names].copy()

In [103]:
# get words!
for ii in range(1, 6):
    bcn = f"w{ii}b"
    cn = f"w{ii}"
    output_df[cn] = output_df[bcn].map(word_byte_to_word_dict)

In [104]:
# count the remainder letter
lc_set = set(ascii_lowercase)
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.update(row[cn])

    return ''.join(lc_set.difference(my_set))

output_df['remaining_letter'] = output_df.apply(get_remainder_letter, axis = 1)



In [105]:
# count unique words - JUST TO VERIFY
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
output_df['n_unique_words'] = output_df[col_names].apply(lambda x: len(set(x)), axis = 1)

In [106]:
# add the words - ALSO TO VERIFY
output_df['bitwise_or'] = 0
output_df['bitwise_and'] = 0
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    output_df[w_cn] = output_df[b_cn].map(word_byte_to_word_dict)
    output_df['bitwise_and'] = output_df['bitwise_and'] & output_df[b_cn]
    output_df['bitwise_or'] = output_df['bitwise_or'] | output_df[b_cn]


In [107]:
output_df.head()

,w1b,w2b,w3b,w4b,w5b,w1,w2,w3,w4,w5,remaining_letter,n_unique_words,bitwise_or,bitwise_and
0,33562898,280068,8914984,19005505,5279872,bizen,jocks,fldxt,gravy,whump,q,5,67043327,0
1,33562898,280068,8914984,5279872,19005505,bizen,jocks,fldxt,whump,gravy,q,5,67043327,0
2,33562898,280068,19005505,5279872,8914984,bizen,jocks,gravy,whump,fldxt,q,5,67043327,0
3,33685523,280068,8914984,18882880,5279872,braze,jocks,fldxt,vying,whump,q,5,67043327,0
4,33685523,280068,8914984,5279872,18882880,braze,jocks,fldxt,whump,vying,q,5,67043327,0


In [108]:
output_df['bitwise_or'].unique().shape

(1,)

# CREATE AND SAVE OUTPUT

In [109]:
output_df.to_excel(excel_writer='test.xlsx', index = False)